In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [2]:
X, y = make_classification(
    n_samples=2500,       
    n_features=20,        
    n_informative=12,      
    n_classes=2,          
    random_state=8       
)

In [3]:
X[:2]

array([[ 0.66393907, -0.08151312, -0.35574504, -2.25523723, -0.82459016,
         0.45469416, -0.24020224, -0.01301753, -2.15150055, -0.20909669,
         1.43943625,  2.27445866,  1.98966981, -0.06946088,  0.07519195,
        -1.39202067,  0.40889453,  1.00180758, -2.70553892,  1.53922232],
       [ 1.93832827,  0.32664886,  1.48643337, -2.02639728, -1.95836646,
         4.80250341,  3.01283631,  0.0772098 ,  1.17200786, -3.12323417,
         0.34412466, -2.89454636,  0.18859416,  6.6140513 , -0.51655982,
         4.92702996, -9.3346991 ,  2.85158828,  0.81600583, -0.58270121]])

In [16]:
y[:2]

array([1, 1])

In [17]:
X.shape

(2500, 20)

In [18]:
y.shape

(2500,)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=8)


In [5]:
X_train.shape

(2000, 20)

In [6]:
X_test.shape

(500, 20)

# Partie 1 : Bagging & Random Forest

## 1.1. Modèle de base

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [8]:
model = DecisionTreeClassifier(random_state=8)

# 5. Train 
model.fit(X_train, y_train)

# 6. predictions
predictions = model.predict(X_test)

# 7. Evaluation
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 83.20%


In [9]:


# 7. Evaluation sur l'ensemble d'entrainement

print(f"Model Accuracy: { accuracy_score(y_train, model.predict(X_train))* 100:.2f}%")

Model Accuracy: 100.00%


## 1.2. Implémentation manuelle du Bagging

In [23]:
bootstraps_index = [pd.DataFrame(X_train).sample(len(X_train), replace=True, random_state=68 + i).index for i in range(50)]

In [24]:
X_train_df = pd.DataFrame(X_train)
y_train_series = pd.Series(y_train)

ens_preds = []
for i, index in enumerate(bootstraps_index):
    
    model = DecisionTreeClassifier(random_state=68 + i) 
    
    
    X_boot = X_train_df.iloc[index]
    y_boot = y_train_series.iloc[index]
    
    
    model.fit(X_boot, y_boot)
    y_pred = model.predict(X_test)
    ens_preds.append(y_pred)

In [29]:
df_preds = pd.DataFrame(ens_preds).T


predictions_finales = df_preds.mode(axis=1)[0]

predictions_finales[:5]

0    1.0
1    0.0
2    1.0
3    0.0
4    0.0
Name: 0, dtype: float64

In [30]:
acc = accuracy_score(y_test, predictions_finales)
print(f"Accuracy: {acc*100:.2f}%")

Accuracy: 88.60%


## 1.3. Random Fores

In [31]:
from sklearn.ensemble import RandomForestClassifier

In [32]:
model_rf = RandomForestClassifier(
    n_estimators=50,
    random_state=8
)

In [33]:
model_rf.fit(X_train, y_train)

print(f"Model Accuracy on train: { accuracy_score(y_train, model_rf.predict(X_train))* 100:.2f}%")
print(f"Model Accuracy on test: { accuracy_score(y_test, model_rf.predict(X_test))* 100:.2f}%")

Model Accuracy on train: 100.00%
Model Accuracy on test: 89.20%


In [38]:
for features in range(1,15):
    model_i = RandomForestClassifier(
    n_estimators=50,
    max_features=features,
    random_state=8
    )

    model_i.fit(X_train, y_train)
    print(f"max_features={features}, Accuracy on test: { accuracy_score(y_test, model_i.predict(X_test))* 100:.2f}% , Accuracy on train: { accuracy_score(y_train, model_i.predict(X_train))* 100:.2f}%,")

max_features=1, Accuracy on test: 86.00% , Accuracy on train: 100.00%,
max_features=2, Accuracy on test: 88.40% , Accuracy on train: 100.00%,
max_features=3, Accuracy on test: 90.00% , Accuracy on train: 100.00%,
max_features=4, Accuracy on test: 89.20% , Accuracy on train: 100.00%,
max_features=5, Accuracy on test: 90.40% , Accuracy on train: 100.00%,
max_features=6, Accuracy on test: 88.80% , Accuracy on train: 100.00%,
max_features=7, Accuracy on test: 89.20% , Accuracy on train: 100.00%,
max_features=8, Accuracy on test: 88.80% , Accuracy on train: 100.00%,
max_features=9, Accuracy on test: 89.60% , Accuracy on train: 100.00%,
max_features=10, Accuracy on test: 90.20% , Accuracy on train: 100.00%,
max_features=11, Accuracy on test: 89.80% , Accuracy on train: 100.00%,
max_features=12, Accuracy on test: 89.00% , Accuracy on train: 100.00%,
max_features=13, Accuracy on test: 89.80% , Accuracy on train: 100.00%,
max_features=14, Accuracy on test: 89.60% , Accuracy on train: 100.00%,


### # **Question**: Pourquoi la limitation des variables à chaque nœud (max_features)améliore-t-elle la diversité des arbres par rapport à un Bagging classique ?

#### # **reponse**: À chaque noeuds la combinaison aleatoire de features inferieure au nombre total de varibles créer des arbres très differentes d'où la diversité ce qui n'est pas le cas du bagging classique qui utilise tous les variables à chaque noeuds

# Partie 2 : Boosting